# Capítulo 8 — Delta-Bar-Delta e EDBD

Códigos do livro *Backpropagation e Redes Neurais* (Araken de Medeiros Santos).

> **Aviso:** este notebook foi gerado a partir dos scripts do livro e **não foi executado como notebook**; as saídas reportadas no texto vêm da execução dos scripts `.py` correspondentes. Execute as células na ordem, a partir desta pasta, com o ambiente de `requirements.txt` (ver Apêndice E). Valores podem diferir ligeiramente em outras plataformas.

In [ ]:
import os, sys
RAIZ = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(RAIZ, 'codigo'))
os.environ.setdefault('FIGS', os.path.join(RAIZ, 'figs'))
print('rnbook em', os.path.join(RAIZ, 'codigo'))

## Script `cap08/cap08.py`

In [ ]:
"""Cap. 8 - Delta-Delta, Delta-Bar-Delta e EDBD: experimentos e figuras (semente fixa; números do texto)."""
import sys, os, time, warnings, itertools
warnings.filterwarnings("ignore")
import numpy as np
from rnbook import MLP
from rnbook import otimizadores as O
from rnbook.estilo import plt, COR, salvar

In [ ]:
# ---------------------------------------------------------------- 8.1.1 derivação da delta-delta (verificação numérica)
# E(w) quadrática 1D: w(t+1) = w(t) - alfa(t) g(t). dE(t+1)/dalfa(t) = -g(t+1) g(t).
lam = 4.0; Eq = lambda w: 0.5 * lam * w * w; gq = lambda w: lam * w
w, alfa, h = 1.0, 0.1, 1e-6
w1 = lambda a: w - a * gq(w)
num = (Eq(w1(alfa + h)) - Eq(w1(alfa - h))) / (2 * h); ana = -gq(w1(alfa)) * gq(w)
print("dE(t+1)/dalfa: numérico %.6f | analítico -g(t+1)g(t) = %.6f" % (num, ana))

In [ ]:
# ---------------------------------------------------------------- quadráticas diagonais: GD, momento, DD, DBD, EDBD
def quad(lam):
    lam = np.array(lam, float); return (lambda v: 0.5 * float(np.sum(lam * v * v))), (lambda v: lam * v)
def itera(otim, lam, x0, lim=1e-12, **kw):
    f, g = quad(lam); r = otim(np.array(x0, float), f, g, tol_grad=0, **kw); L = np.array(r.historico["perda"]); thr = 0.5 * min(lam) * lim
    return (int(np.argmax(L < thr)) if (L < thr).any() else None), r
configs = [("Gradiente descendente ($\\eta=1/\\lambda_{\\max}$)", O.gradiente_descendente, lambda lm: dict(taxa=1 / max(lm), max_iter=20000), COR["gry"]),
           ("Gradiente descendente ($\\eta^*=2/(\\lambda_{\\min}+\\lambda_{\\max})$)", O.gradiente_descendente, lambda lm: dict(taxa=2 / (1 + max(lm)), max_iter=20000), COR["amb"]),
           ("GD com momento ($\\beta=0{,}9$)", O.gradiente_descendente, lambda lm: dict(taxa=0.5 / max(lm), momento=0.9, max_iter=20000), COR["red"]),
           ("Delta-delta", O.delta_delta, lambda lm: dict(alfa0=0.01, gama=1e-3, max_iter=5000), COR["vio"]),
           ("Delta-Bar-Delta", O.delta_bar_delta, lambda lm: dict(alfa0=0.01, kappa=0.005, phi=0.2, teta=0.7, max_iter=5000), COR["ind"]),
           ("EDBD (didático)", O.delta_bar_delta_estendido, lambda lm: dict(alfa0=0.05, kappa_a=0.05, phi_a=0.3, alfa_max=1.0, kappa_m=0.05, mu_max=0.9, max_iter=5000), COR["grn"])]
for lamv in ([1, 10], [1, 10, 100, 1000]):
    print("quadrática diagonal, autovalores", lamv, "(condição %g): iterações até f < 0.5*lam_min*1e-12" % (max(lamv) / min(lamv)))
    for nome, otim, kw, _ in configs:
        it, r = itera(otim, lamv, np.ones(len(lamv)), **kw(lamv)); print("   %-60s %s" % (nome.replace("$", ""), it))
# trajetórias 2D e taxas
lamv = [1, 10]; f, g = quad(lamv)
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.9)); gx, gy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-1.6, 1.6, 200)); ax[0].contour(gx, gy, 0.5 * (gx ** 2 + 10 * gy ** 2), levels=[0.5, 2, 5, 10, 16, 25], colors="#CBD5E1", linewidths=0.8)
def traj(otim, kw, x0=(2.5, 1.2), n=60):
    pts = [np.array(x0)]; th = np.array(x0, float)
    # reexecuta passo a passo: usa max_iter=k para k = 1..n (barato aqui)
    for k in range(1, n + 1):
        r = otim(np.array(x0, float), f, g, tol_grad=0, **{**kw, "max_iter": k}); pts.append(r.theta.copy())
    return np.array(pts)
for nome, otim, kw, c in [configs[1], configs[4], configs[5]]:
    P = traj(otim, kw(lamv), n=40); ax[0].plot(P[:, 0], P[:, 1], "o-", ms=2.5, lw=1.1, color=c, label=nome)
ax[0].set_xlim(-3, 3); ax[0].set_ylim(-1.6, 1.6); ax[0].legend(fontsize=7); ax[0].set_title("Trajetórias na calha $\\frac{1}{2}(x^2+10y^2)$"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
_, rd = itera(O.delta_bar_delta, [1, 10], [1, 1], **configs[4][2]([1, 10]))
fd = O.delta_bar_delta(np.array([2.5, 1.2]), f, g, alfa0=0.01, kappa=0.005, phi=0.2, teta=0.7, max_iter=80, tol_grad=0)
# taxas individuais: executar com 2 pesos e guardar alfa por coordenada (reconstrução manual)
th = np.array([2.5, 1.2]); alf = np.full(2, 0.01); gb = np.zeros(2); A = []
for k in range(80):
    gg = g(th); prod = gb * gg; alf = np.where(prod > 0, alf + 0.005, np.where(prod < 0, alf * 0.8, alf)); th = th - alf * gg; gb = 0.3 * gg + 0.7 * gb; A.append(alf.copy())
A = np.array(A); ax[1].plot(A[:, 0], color=COR["ind"], label="$\\alpha_x$ (curvatura 1)"); ax[1].plot(A[:, 1], color=COR["red"], label="$\\alpha_y$ (curvatura 10)"); ax[1].axhline(2 / 11, color=COR["gry"], ls=":"); ax[1].text(45, 0.19, "$\\eta^*=2/11$", color=COR["gry"], fontsize=8)
ax[1].set_xlabel("iteração"); ax[1].set_ylabel("taxa individual $\\alpha_i$"); ax[1].legend(fontsize=8); ax[1].set_title("DBD: cada coordenada encontra sua taxa"); salvar(fig, "f8_quadratica")
print("DBD na calha: taxas após 80 passos: alfa_x = %.3f, alfa_y = %.3f | (1/lambda_x = 1, 1/lambda_y = 0.1)" % (A[-1, 0], A[-1, 1]))

In [ ]:
# ---------------------------------------------------------------- delta-delta: sensibilidade a gama (falhas)
fig, ax = plt.subplots(1, 2, figsize=(10.2, 3.6)); lamv = [1, 10]; f, g = quad(lamv)
for gama, c in [(1e-3, COR["grn"]), (1e-2, COR["amb"]), (1e-1, COR["red"])]:
    th = np.array([2.5, 1.2]); alf = np.full(2, 0.01); ga = np.zeros(2); A = []; F = []
    for k in range(60):
        gg = g(th); alf = alf + gama * gg * ga; th = th - alf * gg; ga = gg; A.append(alf.copy()); F.append(f(th))
    A = np.array(A); F = np.array(F); ax[0].plot(A[:, 1], color=c, label=f"$\\gamma={gama:g}$"); ax[1].semilogy(np.maximum(F, 1e-300), color=c, label=f"$\\gamma={gama:g}$")
    print("delta-delta gama=%g: alfa_y após 60 passos %.3g | f final %.3g | alfa_y mín %.3g máx %.3g" % (gama, A[-1, 1], F[-1], A[:, 1].min(), A[:, 1].max()))
ax[0].set_ylim(-2, 2); ax[0].axhline(0, color=COR["gry"], lw=0.8); ax[0].set_xlabel("iteração"); ax[0].set_ylabel("$\\alpha_y$"); ax[0].legend(fontsize=8); ax[0].set_title("Delta-delta: a taxa pode ficar negativa ou explodir")
ax[1].set_xlabel("iteração"); ax[1].set_ylabel("$f$ (log)"); ax[1].legend(fontsize=8); ax[1].set_title("Convergência ou divergência conforme $\\gamma$"); salvar(fig, "f8_delta_delta")

In [ ]:
# ---------------------------------------------------------------- rede 1-8-1: protocolo com sementes de ajuste (0-4) e avaliação (5-14)
N = 100; rng = np.random.default_rng(3); X = rng.uniform(0, 1, (N, 1)); T = np.sin(2 * np.pi * X) + rng.normal(0, 0.15, (N, 1))
def run(otim, seed, it=300, **kw):
    r = MLP([1, 8, 1], ["tanh", "identidade"], semente=seed); t0 = time.perf_counter()
    res = otim(r.theta, lambda th: r.perda(th, X, T) / N, lambda th: r.grad(th, X, T) / N, tol_grad=0, max_iter=it, **kw); return res, time.perf_counter() - t0
def mediana(otim, seeds, **kw):
    v = []
    for s in seeds:
        try: x = run(otim, s, **kw)[0].historico["perda"][-1]
        except Exception: x = np.inf
        v.append(x if np.isfinite(x) else 1e9)
    return float(np.median(v)), float(max(v))
tune, aval = range(5), range(5, 15)
best = {}
best["GD"] = min([dict(taxa=e) for e in (0.2, 0.5, 1.0, 1.5)], key=lambda kw: mediana(O.gradiente_descendente, tune, **kw)[0])
best["GD+momento"] = min([dict(taxa=e, momento=b) for e in (0.05, 0.1, 0.2, 0.3) for b in (0.5, 0.9)], key=lambda kw: mediana(O.gradiente_descendente, tune, **kw)[0])
best["Delta-delta"] = min([dict(alfa0=a, gama=gm) for a in (0.05, 0.1, 0.3) for gm in (1e-3, 1e-2, 1e-1)], key=lambda kw: mediana(O.delta_delta, tune, **kw)[0])
best["DBD"] = min([dict(alfa0=a, kappa=k, phi=p, teta=t) for a, k, p, t in itertools.product((0.1, 0.3), (0.01, 0.05, 0.2), (0.1, 0.3), (0.5, 0.9))], key=lambda kw: mediana(O.delta_bar_delta, tune, **kw)[0])
best["EDBD"] = min([dict(alfa0=a, kappa_a=ka, phi_a=pa, alfa_max=am, kappa_m=km, mu_max=mm) for a, ka, pa, am, km, mm in itertools.product((0.1, 0.3), (0.05, 0.2), (0.1, 0.3), (2.0, 5.0), (0.01, 0.05), (0.5, 0.9))], key=lambda kw: mediana(O.delta_bar_delta_estendido, tune, **kw)[0])
ots = {"GD": O.gradiente_descendente, "GD+momento": O.gradiente_descendente, "Delta-delta": O.delta_delta, "DBD": O.delta_bar_delta, "EDBD": O.delta_bar_delta_estendido}
for k, v in best.items(): print("melhor (sementes 0-4) %-12s %s" % (k, v))
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.8)); cores = {"GD": COR["gry"], "GD+momento": COR["red"], "Delta-delta": COR["vio"], "DBD": COR["ind"], "EDBD": COR["grn"]}; finais = {}
for nome in ots:
    H = []; F = []; tt = []
    for s in aval:
        res, dt = run(ots[nome], s, **best[nome]); h = res.historico["perda"]; H.append(h); F.append(h[-1] if np.isfinite(h[-1]) else 1e9); tt.append(dt)
    def _fix(h, n=None):
        h = np.asarray(h, float); fin = np.isfinite(h)
        ult = h[fin][-1] if fin.any() else 1e9
        h = np.where(fin, h, ult)
        return h
    L = max(len(h) for h in H)
    H = np.array([np.concatenate([_fix(h), np.full(L - len(h), _fix(h)[-1])]) for h in H]); Hm = np.median(H, axis=0); finais[nome] = F
    ax[0].semilogy(Hm, color=cores[nome], label=nome); cruz = int(np.argmax(Hm < 0.03)) if (Hm < 0.03).any() else None
    print("avaliação (sementes 5-14) %-12s perda final mediana %.4f [mín %.4f, máx %.4f] | época em que a mediana cruza 0.03: %s | tempo %.3f s" % (nome, np.median(F), min(F), max(F), cruz, np.mean(tt)))
ax[0].axhline(0.5 * 0.15 ** 2, color=COR["gry"], ls=":"); ax[0].legend(fontsize=8); ax[0].set_xlabel("época"); ax[0].set_ylabel("perda $E/N$ (mediana de 10 sementes)"); ax[0].set_title("Rede 1–8–1 (hiperparâmetros ajustados em outras sementes)")
ax[1].boxplot([finais[n] for n in ots], tick_labels=[n.replace("Delta-delta", "Delta-\ndelta").replace("GD+momento", "GD +\nmomento") for n in ots]); ax[1].set_yscale("log"); ax[1].set_ylabel("perda final $E/N$"); ax[1].set_title("Dispersão da perda final")
salvar(fig, "f8_rede")
# taxas individuais da DBD na rede
res, _ = run(O.delta_bar_delta, 5, **best["DBD"]); print("DBD (semente 5): alfa mín %.3g | mediana %.3g | máx %.3g na última época; alfa0 = %g" % (res.historico["alfa_min"][-1], res.historico["alfa_med"][-1], res.historico["alfa_max"][-1], best["DBD"]["alfa0"]))
res, _ = run(O.delta_bar_delta_estendido, 5, **best["EDBD"]); print("EDBD (semente 5): alfa mín %.3g | mediana %.3g | máx %.3g; momento mediano %.3g" % (res.historico["alfa_min"][-1], res.historico["alfa_med"][-1], res.historico["alfa_max"][-1], res.historico["mom_med"][-1]))
fig, ax = plt.subplots(1, 2, figsize=(10.2, 3.5)); r1, _ = run(O.delta_bar_delta, 5, **best["DBD"]); r2, _ = run(O.delta_bar_delta_estendido, 5, **best["EDBD"])
for r_, a_, tit in [(r1, ax[0], "DBD"), (r2, ax[1], "EDBD")]:
    a_.semilogy(r_.historico["alfa_min"], color=COR["gry"], label="$\\min\\alpha_i$"); a_.semilogy(r_.historico["alfa_med"], color=COR["ind"], label="mediana"); a_.semilogy(r_.historico["alfa_max"], color=COR["red"], label="$\\max\\alpha_i$"); a_.set_xlabel("época"); a_.set_ylabel("taxa individual"); a_.legend(fontsize=8); a_.set_title(f"{tit}: taxas individuais (semente 5)")
salvar(fig, "f8_taxas")

In [ ]:
# ---------------------------------------------------------------- sensibilidade da DBD (kappa x phi) em sementes de avaliação
kap = (0.005, 0.02, 0.05, 0.2); phs = (0.05, 0.1, 0.3, 0.5); G = np.zeros((len(phs), len(kap)))
for i, p in enumerate(phs):
    for j, k in enumerate(kap): G[i, j] = mediana(O.delta_bar_delta, aval, alfa0=0.1, kappa=k, phi=p, teta=0.9)[0]
print("DBD, sensibilidade (alfa0=0.1, teta=0.9), mediana da perda final em 10 sementes; linhas phi, colunas kappa", kap); [print("  phi=%.2f:" % p, np.round(G[i], 4)) for i, p in enumerate(phs)]
tets = (0.0, 0.3, 0.7, 0.9, 0.99); S = [mediana(O.delta_bar_delta, aval, alfa0=0.1, kappa=0.05, phi=0.1, teta=t)[0] for t in tets]; print("DBD, efeito de teta (alfa0=0.1, kappa=0.05, phi=0.1):", dict(zip(tets, np.round(S, 4))))
fig, ax = plt.subplots(1, 2, figsize=(10.2, 3.6)); im = ax[0].imshow(np.log10(G), cmap="viridis_r", aspect="auto"); ax[0].set_xticks(range(len(kap))); ax[0].set_xticklabels(kap); ax[0].set_yticks(range(len(phs))); ax[0].set_yticklabels(phs); ax[0].set_xlabel("$\\kappa$ (aumento aditivo)"); ax[0].set_ylabel("$\\phi$ (redução)")
for i in range(len(phs)):
    for j in range(len(kap)): ax[0].text(j, i, "%.3f" % G[i, j] if G[i, j] < 1e3 else "div.", ha="center", va="center", color="white", fontsize=8)
ax[0].set_title("DBD: perda final mediana ($\\log_{10}$ em cor)"); ax[1].semilogy(range(len(tets)), S, "o-", color=COR["ind"]); ax[1].set_xticks(range(len(tets))); ax[1].set_xticklabels(tets); ax[1].set_xlabel("$\\theta$ (peso da memória do gradiente)"); ax[1].set_ylabel("perda final mediana"); ax[1].set_title("DBD: efeito de $\\theta$"); salvar(fig, "f8_sensibilidade")

In [ ]:
# ---------------------------------------------------------------- mecanismo: g, g_bar e produto em um peso
r = MLP([1, 8, 1], ["tanh", "identidade"], semente=5); th = r.theta.copy(); alf = np.full_like(th, 0.1); gb = np.zeros_like(th); reg = []
for k in range(120):
    gg = r.grad(th, X, T) / N; prod = gb * gg; alf = np.where(prod > 0, alf + 0.05, np.where(prod < 0, alf * 0.9, alf)); th = th - alf * gg; gb = 0.1 * gg + 0.9 * gb; reg.append((gg[3], gb[3], np.sign(prod[3]), alf[3]))
reg = np.array(reg); fig, ax = plt.subplots(2, 1, figsize=(7.6, 4.6), sharex=True); ax[0].plot(reg[:, 0], color=COR["gry"], lw=1, label="$g(t)$ (gradiente)"); ax[0].plot(reg[:, 1], color=COR["ind"], lw=2, label="$\\bar g(t)$ (média exponencial)"); ax[0].legend(fontsize=8); ax[0].set_ylabel("derivada do peso 4")
ax[1].plot(reg[:, 3], color=COR["red"]); ax[1].set_ylabel("taxa $\\alpha_4$"); ax[1].set_xlabel("época"); salvar(fig, "f8_mecanismo")
print("OK cap08")

## Exercícios práticos e projeto: soluções de referência

### 1

Implemente o DBD em NumPy (sem o pacote `rnbook`), aplique-o à quadrática $f(x,y)=\tfrac12(x^2+10y^2)$ a partir de $(2{,}5;\,1{,}2)$ e compare com o gradiente descendente de passo $\eta^*=2/11$.

1. Escreva `dbd(theta0, grad, alfa0, kappa, phi, teta, max_iter)` vetorizada, que devolva o histórico de $\theta$ e das taxas.
2. Use $\alpha_0=0{,}01$, $\kappa=0{,}005$, $\phi=0{,}2$, $\theta=0{,}7$.
3. Conte as iterações até $f<10^{-12}$ nos dois métodos e imprima as taxas individuais após 20, 40 e 79 passos.

**Pergunta:** por que as duas taxas terminam tão diferentes uma da outra?

In [ ]:
import numpy as np

lam = np.array([1.0, 10.0])
f = lambda th: 0.5 * np.sum(lam * th ** 2)
grad = lambda th: lam * th


def dbd(theta0, grad, alfa0, kappa, phi, teta, max_iter):
    theta = theta0.copy()
    alfa = np.full_like(theta, alfa0)
    gbar = np.zeros_like(theta)
    hist_th, hist_al = [theta.copy()], []
    for _ in range(max_iter):
        g = grad(theta)
        prod = gbar * g                                    # teste de sinal com gbar(t-1)
        alfa = np.where(prod > 0, alfa + kappa, np.where(prod < 0, alfa * (1 - phi), alfa))
        theta = theta - alfa * g
        gbar = (1 - teta) * g + teta * gbar                # memória atualizada por último
        hist_th.append(theta.copy()); hist_al.append(alfa.copy())
    return np.array(hist_th), np.array(hist_al)


th0 = np.array([2.5, 1.2])
H, A = dbd(th0, grad, 0.01, 0.005, 0.2, 0.7, 400)
it_dbd = next(k for k, t in enumerate(H) if f(t) < 1e-12)

th = th0.copy(); it_gd = None
for k in range(1, 2000):
    th = th - (2 / 11) * grad(th)
    if f(th) < 1e-12:
        it_gd = k; break

print("iterações até f < 1e-12: DBD =", it_dbd, "| GD (eta* = 2/11) =", it_gd)
for k in (20, 40, 79):
    print("taxas após %d passos: alfa_x = %.4f, alfa_y = %.2e" % (k, A[k - 1, 0], A[k - 1, 1]))

### 2

Verifique, em uma quadrática, que a delta-delta depende da **escala** do erro e que o DBD, que testa apenas o sinal, depende menos.

1. Tome $E_c(\theta)=c\cdot\tfrac12(x^2+10y^2)$ com $c\in\{0{,}1;\,1;\,10\}$ e ponto inicial $(2{,}5;\,1{,}2)$.
2. Rode 60 passos da delta-delta ($\alpha_0=0{,}01$, $\gamma=10^{-3}$) e do DBD ($\alpha_0=0{,}01$, $\kappa=0{,}005$, $\phi=0{,}2$, $\theta=0{,}7$), usando o gradiente $c\,\nabla f$.
3. Registre $E_c/c$ (isto é, $f$) ao final e se houve valores não finitos.

**Pergunta:** o que muda e o que não muda quando $c$ cresce? Por quê?

In [ ]:
import numpy as np

lam = np.array([1.0, 10.0])
f = lambda th: 0.5 * np.sum(lam * th ** 2)


def delta_delta(th0, c, alfa0=0.01, gama=1e-3, n=60):
    th, alfa, g_ant = th0.copy(), np.full(2, alfa0), np.zeros(2)
    for _ in range(n):
        g = c * lam * th
        alfa = alfa + gama * g * g_ant
        th = th - alfa * g
        g_ant = g
        if not np.all(np.isfinite(th)):
            return th, alfa
    return th, alfa


def dbd(th0, c, alfa0=0.01, kappa=0.005, phi=0.2, teta=0.7, n=60):
    th, alfa, gbar = th0.copy(), np.full(2, alfa0), np.zeros(2)
    for _ in range(n):
        g = c * lam * th
        prod = gbar * g
        alfa = np.where(prod > 0, alfa + kappa, np.where(prod < 0, alfa * (1 - phi), alfa))
        th = th - alfa * g
        gbar = (1 - teta) * g + teta * gbar
        if not np.all(np.isfinite(th)):
            return th, alfa
    return th, alfa


th0 = np.array([2.5, 1.2])
for c in (0.1, 1.0, 10.0):
    thd, ad = delta_delta(th0, c)
    thb, ab = dbd(th0, c)
    print("c=%-5g delta-delta: f=%.3e (finito: %s), alfa=%s | DBD: f=%.3e (finito: %s), alfa=%s"
          % (c, f(thd), np.all(np.isfinite(thd)), np.round(ad, 4), f(thb), np.all(np.isfinite(thb)), np.round(ab, 4)))

### 3

Compare gradiente descendente com momento, DBD, EDBD e iRPROP− (pacote `rnbook`) na base **Breast Cancer Wisconsin (Diagnostic)** (UCI; embutida no scikit-learn), escolhendo hiperparâmetros **só na validação**.

1. Separe 60/20/20 estratificado (`random_state=0`) em treino, validação e teste; padronize com o treino.
2. Rede 30–10–1 (tanh, saída sigmoide), erro quadrático médio, 200 épocas. Para cada método, teste uma grade pequena (4 a 8 combinações), com a semente 0, e escolha a de menor perda de validação.
3. Com a configuração escolhida, treine com as sementes 1 a 5 e relate a acurácia de teste (mediana e intervalo) e o tempo.

**Pergunta:** a ordem dos métodos é a mesma na validação e no teste? O que se pode concluir?

In [ ]:
import time
import warnings
import itertools
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from rnbook import MLP, otimizadores as O

warnings.filterwarnings("ignore")
X, y = load_breast_cancer(return_X_y=True)
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=0, stratify=y_tmp)
esc = StandardScaler().fit(X_tr)
Xtr, Xva, Xte = esc.transform(X_tr), esc.transform(X_va), esc.transform(X_te)
Ttr, Tva = y_tr[:, None].astype(float), y_va[:, None].astype(float)
N = len(y_tr)


def treinar(otim, semente, **kw):
    r = MLP([30, 10, 1], ["tanh", "sigmoide"], semente=semente)
    t0 = time.perf_counter()
    res = otim(r.theta, lambda th: r.perda(th, Xtr, Ttr) / N, lambda th: r.grad(th, Xtr, Ttr) / N,
               max_iter=200, tol_grad=0, **kw)
    return r, res, time.perf_counter() - t0


def perda_val(r, th):
    return float(np.mean((r.prever(Xva, th) - Tva) ** 2) / 2)


def acc(r, th, Xa, ya):
    return float(np.mean((r.prever(Xa, th).ravel() > 0.5) == ya))


grades = {
    "GD+momento": (O.gradiente_descendente, [dict(taxa=a, momento=b) for a, b in itertools.product((0.1, 0.5), (0.5, 0.9))]),
    "DBD": (O.delta_bar_delta, [dict(alfa0=0.1, kappa=k, phi=p, teta=0.7) for k, p in itertools.product((0.01, 0.05), (0.1, 0.3))]),
    "EDBD": (O.delta_bar_delta_estendido, [dict(alfa0=0.1, kappa_a=k, phi_a=0.3, alfa_max=a, kappa_m=0.05, mu_max=0.9)
                                          for k, a in itertools.product((0.02, 0.1), (1.0, 3.0))]),
    "iRPROP-": (O.rprop, [dict(variante="irprop-", delta0=d) for d in (0.01, 0.05, 0.1, 0.2)]),
}
for nome, (otim, grade) in grades.items():
    sc = []
    for kw in grade:
        try:
            r, res, _ = treinar(otim, 0, **kw); v = perda_val(r, res.theta)
        except Exception:
            v = np.inf
        sc.append(v if np.isfinite(v) else np.inf)
    kw = grade[int(np.argmin(sc))]
    ac_va, ac_te, tt = [], [], []
    for s in range(1, 6):
        r, res, dt = treinar(otim, s, **kw)
        ac_va.append(acc(r, res.theta, Xva, y_va)); ac_te.append(acc(r, res.theta, Xte, y_te)); tt.append(dt)
    print("%-11s melhor %s\n            val: %.4f [%.4f; %.4f] | teste: %.4f [%.4f; %.4f] | tempo %.3f s"
          % (nome, {k: v for k, v in kw.items()}, np.median(ac_va), min(ac_va), max(ac_va),
             np.median(ac_te), min(ac_te), max(ac_te), np.mean(tt)))

### Otimizadores adaptativos em regressão: Friedman #1

**Objetivo.** Comparar, com protocolo honesto, os treinadores do livro (gradiente descendente com momento, DBD, EDBD, iRPROP− e LM) em um problema de regressão, medindo desempenho em teste, tempo e robustez entre sementes.

**Dados.** `make_friedman1` do scikit-learn (600 amostras, 10 atributos, `noise=1.0`, `random_state=0`; só 5 atributos são informativos).

**Tarefas.**
1. Separe 60/20/20 (`random_state=0`); padronize os atributos com o treino e deixe o alvo na escala original dividido pelo desvio do treino (registre).
2. Rede 10–10–1 (tanh, saída linear). Para cada método, escolha os hiperparâmetros em uma grade de 4 combinações (semente de ajuste 100) pela perda de **validação**.
3. Com a configuração escolhida, treine com 10 sementes (0 a 9), com 300 épocas (LM: 60 iterações), e registre o $R^2$ de teste (mediana e intervalo), o tempo e o número de execuções divergentes.
4. Redija 10 a 12 linhas de conclusão, apontando o que o experimento **não** permite afirmar.

**Para ir além (não testado no livro).** Repita com `fetch_california_housing` (Kaggle/UCI; exige internet) e com minilotes para o gradiente descendente; discuta por que o DBD não é projetado para esse regime.

In [ ]:
import time
import warnings
import itertools
import numpy as np
from sklearn.datasets import make_friedman1
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from rnbook import MLP, otimizadores as O

warnings.filterwarnings("ignore")
X, y = make_friedman1(n_samples=600, n_features=10, noise=1.0, random_state=0)
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=0)
esc = StandardScaler().fit(X_tr)
Xtr, Xva, Xte = esc.transform(X_tr), esc.transform(X_va), esc.transform(X_te)
dp = y_tr.std()
Ttr, Tva = (y_tr / dp)[:, None], (y_va / dp)[:, None]
N = len(y_tr)
print("desvio do alvo de treino (divisor):", round(float(dp), 4), "| N treino =", N)


def treinar(metodo, semente, kw):
    r = MLP([10, 10, 1], ["tanh", "identidade"], semente=semente)
    t0 = time.perf_counter()
    if metodo == "LM":
        res = O.levenberg_marquardt(r.theta, lambda th: r.residuos(th, Xtr, Ttr), lambda th: r.jacobiana(th, Xtr),
                                    max_iter=60, tol_grad=0, **kw)
    else:
        otim = {"GD+momento": O.gradiente_descendente, "DBD": O.delta_bar_delta,
                "EDBD": O.delta_bar_delta_estendido, "iRPROP-": O.rprop}[metodo]
        res = otim(r.theta, lambda th: r.perda(th, Xtr, Ttr) / N, lambda th: r.grad(th, Xtr, Ttr) / N,
                   max_iter=300, tol_grad=0, **kw)
    return r, res.theta, time.perf_counter() - t0


def r2(r, th, Xa, ya):
    return r2_score(ya, r.prever(Xa, th).ravel() * dp)


grades = {
    "GD+momento": [dict(taxa=a, momento=b) for a, b in itertools.product((0.05, 0.2), (0.5, 0.9))],
    "DBD": [dict(alfa0=0.1, kappa=k, phi=p, teta=0.7) for k, p in itertools.product((0.01, 0.05), (0.1, 0.3))],
    "EDBD": [dict(alfa0=0.1, kappa_a=k, phi_a=0.3, alfa_max=a, kappa_m=0.05, mu_max=0.9)
             for k, a in itertools.product((0.02, 0.1), (1.0, 3.0))],
    "iRPROP-": [dict(variante="irprop-", delta0=d) for d in (0.01, 0.05, 0.1, 0.2)],
    "LM": [dict(mu0=m, mu_inc=i) for m, i in itertools.product((1e-3, 1e-1), (10.0, 5.0))],
}
for metodo, grade in grades.items():
    sc = []
    for kw in grade:
        try:
            r, th, _ = treinar(metodo, 100, kw); v = float(np.mean((r.prever(Xva, th) - Tva) ** 2))
        except Exception:
            v = np.inf
        sc.append(v if np.isfinite(v) else np.inf)
    kw = grade[int(np.argmin(sc))]
    va, te, tt, div = [], [], [], 0
    for s in range(10):
        r, th, dt = treinar(metodo, s, kw)
        a, b = r2(r, th, Xva, y_va), r2(r, th, Xte, y_te)
        if not np.isfinite(b): div += 1; continue
        va.append(a); te.append(b); tt.append(dt)
    print("%-11s %s\n            R2 val %.3f | R2 teste %.3f [%.3f; %.3f] | tempo %.3f s | divergências: %d"
          % (metodo, kw, np.median(va), np.median(te), min(te), max(te), np.mean(tt), div))

## Projeto integrador final

### Comparação de treinadores em classificação e regressão

**Objetivo.** Integrar o que foi visto nos oito capítulos: treinar a mesma rede com os algoritmos do livro e comparar desempenho em dados não vistos, custo e robustez, com um protocolo que evite vazamento e conclusões infundadas.

**Dados (embutidos no scikit-learn; originários do repositório UCI).**
- Classificação: *Breast Cancer Wisconsin (Diagnostic)* (569 amostras, 30 atributos).
- Regressão: *Diabetes* (442 amostras, 10 atributos; alvo = progressão da doença).

**Algoritmos.** Gradiente descendente com momento, Quickprop, iRPROP−, DBD, EDBD e Levenberg–Marquardt (pacote `rnbook`), mais o `lbfgs` do scikit-learn como referência externa. O perceptron e o Adam **não** entram: o primeiro não resolve regressão, e o segundo não é implementado no livro.

**Tarefas.**
1. Em cada problema, separe 60/20/20 (treino/validação/teste; estratificado na classificação), `random_state=0`; padronize com o treino. Na regressão, divida o alvo pelo desvio do treino.
2. Rede com uma camada oculta de 8 neurônios (tanh); saída sigmoide (classificação) ou linear (regressão); erro quadrático.
3. Para cada algoritmo, escolha hiperparâmetros em uma grade pequena, com uma **semente de ajuste** (100), pela perda de validação.
4. Com a configuração escolhida, treine com as sementes 0 a 4 e registre: desempenho de teste (acurácia ou $R^2$; mediana e intervalo), número de épocas ou iterações, tempo e divergências.
5. Faça um quadro-resumo e redija 12 a 15 linhas: quais afirmações o experimento sustenta, quais **não** sustenta e que mudanças tornariam a comparação mais confiável (mais sementes, validação cruzada, mais bases, teste estatístico, regularização, parada antecipada).

**Para ir além (não executado no livro).** Repita com bases da UCI/Kaggle ainda não vistas (por exemplo, *Wine Quality* e *Heart Disease*), que exigem acesso à internet; inclua a validação cruzada aninhada; e inclua um algoritmo de minilote (`solver="adam"` do scikit-learn) como contraste, comentando por que ele responde a outro regime.

In [ ]:
import time
import warnings
import itertools
import numpy as np
import sklearn
import scipy
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from rnbook import MLP, otimizadores as O

warnings.filterwarnings("ignore")
print("versões: numpy", np.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)


def preparar(X, y, classif):
    est = y if classif else None
    X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=est)
    est = y_tmp if classif else None
    X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=0, stratify=est)
    esc = StandardScaler().fit(X_tr)
    dp = 1.0 if classif else float(y_tr.std())
    return dict(Xtr=esc.transform(X_tr), Xva=esc.transform(X_va), Xte=esc.transform(X_te),
                ytr=y_tr, yva=y_va, yte=y_te, T=(y_tr / dp)[:, None].astype(float), Tva=(y_va / dp)[:, None].astype(float), dp=dp)


def metodo_rodar(metodo, D, semente, kw, classif):
    ativ = ["tanh", "sigmoide"] if classif else ["tanh", "identidade"]
    r = MLP([D["Xtr"].shape[1], 8, 1], ativ, semente=semente)
    N = len(D["T"])
    t0 = time.perf_counter()
    if metodo == "LM":
        res = O.levenberg_marquardt(r.theta, lambda th: r.residuos(th, D["Xtr"], D["T"]),
                                    lambda th: r.jacobiana(th, D["Xtr"]), max_iter=60, tol_grad=0, **kw)
    else:
        otim = {"GD+momento": O.gradiente_descendente, "Quickprop": O.quickprop, "iRPROP-": O.rprop,
                "DBD": O.delta_bar_delta, "EDBD": O.delta_bar_delta_estendido}[metodo]
        res = otim(r.theta, lambda th: r.perda(th, D["Xtr"], D["T"]) / N, lambda th: r.grad(th, D["Xtr"], D["T"]) / N,
                   max_iter=300, tol_grad=0, **kw)
    return r, res, time.perf_counter() - t0


def medida(r, th, D, parte, classif):
    X, y = D["X" + parte], D["y" + parte]
    p = r.prever(X, th).ravel()
    return float(np.mean((p > 0.5) == y)) if classif else float(r2_score(y, p * D["dp"]))


grades = {
    "GD+momento": [dict(taxa=a, momento=b) for a, b in itertools.product((0.05, 0.3), (0.5, 0.9))],
    "Quickprop": [dict(epsilon=e, mu=1.75) for e in (0.01, 0.05, 0.1, 0.5)],
    "iRPROP-": [dict(variante="irprop-", delta0=d) for d in (0.01, 0.05, 0.1, 0.2)],
    "DBD": [dict(alfa0=0.1, kappa=k, phi=p, teta=0.7) for k, p in itertools.product((0.01, 0.05), (0.1, 0.3))],
    "EDBD": [dict(alfa0=0.1, kappa_a=k, phi_a=0.3, alfa_max=a, kappa_m=0.05, mu_max=0.9) for k, a in itertools.product((0.02, 0.1), (1.0, 3.0))],
    "LM": [dict(mu0=m, mu_inc=i) for m, i in itertools.product((1e-3, 1e-1), (10.0, 5.0))],
}
problemas = {"Breast Cancer (classificação, acurácia)": (load_breast_cancer(return_X_y=True), True),
             "Diabetes (regressão, R2)": (load_diabetes(return_X_y=True), False)}
for nome_p, ((X, y), classif) in problemas.items():
    D = preparar(X, y, classif)
    print("\n== %s | treino %d, validação %d, teste %d" % (nome_p, len(D["ytr"]), len(D["yva"]), len(D["yte"])))
    for metodo, grade in grades.items():
        sc = []
        for kw in grade:
            try:
                r, res, _ = metodo_rodar(metodo, D, 100, kw, classif)
                v = float(np.mean((r.prever(D["Xva"], res.theta) - D["Tva"]) ** 2))
            except Exception:
                v = np.inf
            sc.append(v if np.isfinite(v) else np.inf)
        kw = grade[int(np.argmin(sc))]
        va, te, tt, it, div = [], [], [], [], 0
        for s in range(5):
            r, res, dt = metodo_rodar(metodo, D, s, kw, classif)
            m_te = medida(r, res.theta, D, "te", classif)
            if not np.isfinite(m_te):
                div += 1; continue
            va.append(medida(r, res.theta, D, "va", classif)); te.append(m_te); tt.append(dt); it.append(res.n_iter)
        print("%-11s %-70s teste %.3f [%.3f; %.3f] | val %.3f | it %d | %.3f s | div %d"
              % (metodo, str(kw)[:70], np.median(te), min(te), max(te), np.median(va), np.median(it), np.mean(tt), div))
    ref_te, ref_tt = [], []
    for s in range(5):
        if classif:
            m = MLPClassifier(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", max_iter=300, random_state=s)
        else:
            m = MLPRegressor(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", max_iter=300, random_state=s)
        t0 = time.perf_counter()
        m.fit(D["Xtr"], D["ytr"] if classif else D["ytr"] / D["dp"])
        ref_tt.append(time.perf_counter() - t0)
        ref_te.append(m.score(D["Xte"], D["yte"]) if classif else r2_score(D["yte"], m.predict(D["Xte"]) * D["dp"]))
    print("%-11s %-70s teste %.3f [%.3f; %.3f] | %.3f s" % ("sk lbfgs", "(referência externa)", np.median(ref_te), min(ref_te), max(ref_te), np.mean(ref_tt)))